### Paths

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

observations_path = Path("/scratch/s1803229/ATL08_parquets/S01.parquet")
experiment = Path("/scratch/s1803229/fsm2_ensemble/S01/REPLACE_WITH_EXPERIMENT_ID")
lookup_path = Path("../../output/classification_uncertainty_EgEv.nc")
output = experiment / "comparison.parquet"

probs = ["prob_no_snow", "prob_ground_snow", "prob_ground_canopy_snow"]

### Creating grouped parquet

In [ ]:
grouped_path = observations_path.with_stem(observations_path.stem + "_grouped")

if grouped_path.exists():
    obs = pd.read_parquet(grouped_path)
else:
    keys = ["site_id", "resource", "date", "lat", "lon"]
    stats = {
        "pvpg": "mean", "pv": "max", "pg": "max", "data_quantity": "max",
        "snr": "mean", "layer_flag": "mean", "msw_flag": "mean",
        "pv_ratio_mean": "mean", "pv_ratio_max": "mean",
    }
    columns = keys + list(stats) + ["beam_str", "outlier", "Eg", "Ev"]
    raw = pd.read_parquet(observations_path, columns=columns)

    aggs = {name: (name, method) for name, method in stats.items()}
    for strength in ("strong", "weak"):
        keep = raw["beam_str"].eq(strength) & raw["outlier"].eq(1)
        for variable in ("Eg", "Ev"):
            name = f"{variable}_{strength}"
            raw[name] = raw[variable].where(keep)
            aggs[name] = (name, "median")

    obs = raw.groupby(keys, sort=False, observed=True).agg(
        **aggs, segment_rows=("Eg", "size")
    ).reset_index()
    del raw
    obs["date"] = pd.to_datetime(obs["date"], format="%d/%m/%Y")
    temporary = grouped_path.with_suffix(".partial.parquet")
    obs.to_parquet(temporary, index=False, compression="zstd")
    temporary.replace(grouped_path)

print(f"{len(obs):,} groups")
obs.head()

### Assign probabilities and nearest model hour

In [ ]:
if "observation_time" in obs:
    obs["observation_time"] = pd.to_datetime(obs["observation_time"], utc=True)
else:
    stamp = obs["resource"].str.extract(r"ATL0[38]_(\d{14})", expand=False)
    obs["observation_time"] = pd.to_datetime(stamp, format="%Y%m%d%H%M%S", utc=True)

if obs["observation_time"].isna().any():
    raise ValueError("Missing observation timestamp")

obs["model_time"] = obs["observation_time"].dt.round("h").dt.tz_localize(None)

query = {
    name: xr.DataArray(obs[name].to_numpy(), dims="observation")
    for name in ("Eg_strong", "Ev_strong")
}
with xr.open_dataset(lookup_path) as lookup:
    for name in probs:
        obs[name] = lookup[name].interp(query).values

valid = np.isfinite(obs[probs]).all(axis=1)
obs["icesat_class"] = pd.array(
    np.where(valid, obs[probs].to_numpy().argmax(axis=1), np.nan), dtype="Int8"
)

### Compare all members and write parquet
ground snow fraction has to be above 50% for binary to be 1, canopy snow has to be above 0.5 kg per meter squared for binary to be 1.

In [ ]:
from contextlib import ExitStack
from tempfile import TemporaryDirectory
import pyarrow as pa
import pyarrow.parquet as pq

members = (experiment / "members.txt").read_text().split()
site = experiment.parent.name
variables = ["ground_snow_cover_fraction", "ground_snow_mass", "canopy_snow_mass"]
cell_hour = ["model_latitude", "model_longitude", "model_time"]

with TemporaryDirectory(dir=experiment) as tmp:
    pending = Path(tmp) / "comparison.parquet"

    with ExitStack() as stack:
        for i, member in enumerate(members):
            path = experiment / member / f"{site}.nc"
            rows = obs.copy()

            with xr.open_dataset(path) as model:
                indices = {}
                inside = np.ones(len(rows), dtype=bool)

                for dim, col in [("latitude", "lat"), ("longitude", "lon")]:
                    centres = model[dim].values
                    spacing = np.abs(np.diff(centres))
                    if not len(spacing) or not np.allclose(spacing, spacing[0]):
                        raise ValueError(f"Expected a regular {dim} grid")

                    idx = model.indexes[dim].get_indexer(rows[col], method="nearest")
                    rows[f"model_{dim}"] = centres[idx]
                    inside &= np.abs(rows[col] - centres[idx]) <= spacing[0] / 2 + 1e-8
                    indices[dim] = idx

                time_idx = model.indexes["time"].get_indexer(rows["model_time"])
                inside &= time_idx >= 0
                indices["time"] = np.maximum(time_idx, 0)

                selected = model[variables].isel({
                    dim: xr.DataArray(idx, dims="observation")
                    for dim, idx in indices.items()
                }).load()
                for name in variables:
                    rows[name] = np.where(inside, selected[name].values, np.nan)

            ground = rows["ground_snow_cover_fraction"] >= 0.5
            canopy = rows["canopy_snow_mass"] > 0.5
            valid_model = np.isfinite(rows[variables]).all(axis=1)
            classes = np.where(ground, np.where(canopy, 2, 1), 0)
            rows["model_class"] = pd.array(
                np.where(valid_model, classes, np.nan), dtype="Int8"
            )

            matched = valid_model & rows["icesat_class"].notna()
            rows["matched"] = matched
            rows["comparison_weight"] = 0.0
            count = rows.loc[matched].groupby(["resource", *cell_hour])["resource"].transform("size")
            rows.loc[matched, "comparison_weight"] = 1 / count
            rows["member"] = member
            rows["model_file"] = str(path)
            rows["ground_threshold_fraction"] = 0.5
            rows["canopy_threshold_kg_m2"] = 0.5

            table = pa.Table.from_pandas(rows, preserve_index=False)
            if i == 0:
                writer = stack.enter_context(pq.ParquetWriter(pending, table.schema))
            writer.write_table(table)
            print(f"{member}: {matched.sum():,} matched groups")

    pending.replace(output)

### Load comparison and rank members

In [ ]:
comparison = pd.read_parquet(output)
comparison["scheme"] = comparison["member"].str.split("_", n=1).str[0]

matched = comparison.loc[comparison["matched"]].copy()
matched["weighted_correct"] = (
    matched["comparison_weight"] * matched["model_class"].eq(matched["icesat_class"])
)

scores = matched.groupby(["scheme", "member"]).agg(
    groups=("matched", "size"),
    correct=("weighted_correct", "sum"),
    weight=("comparison_weight", "sum"),
)
scores["agreement_pct"] = 100 * scores["correct"] / scores["weight"]
scores[["groups", "agreement_pct"]].sort_values("agreement_pct", ascending=False).round(2)

### Confusion Matrix

In [ ]:
member = "hp98_cap1_loss1"
one = matched.loc[matched["member"].eq(member)]

matrix = pd.crosstab(
    one["icesat_class"], one["model_class"],
    values=one["comparison_weight"], aggfunc="sum",
).reindex(index=[0, 1, 2], columns=[0, 1, 2]).fillna(0)

matrix.div(matrix.sum(axis=1), axis=0).mul(100).round(1)

### Inspect 1km cells sharing same cell-hour

In [ ]:
cell_hour = ["model_latitude", "model_longitude", "model_time"]
groups = one.groupby(cell_hour)
counts = groups.size().sort_values(ascending=False)
display(counts.head())

if len(counts):
    display(groups.get_group(counts.index[0])[
        ["resource", "lat", "lon", *probs, "icesat_class", "model_class",
         "ground_snow_cover_fraction", "canopy_snow_mass", "comparison_weight"]
    ])